In [1]:
import pandas as pd

df = pd.read_csv("../data/results.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (49547, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


In [2]:
import pandas as pd

r = pd.read_csv("c:\\Users\\PC\\Downloads\\nations_league_2026_27.csv")
r.head()

,date,group,home_team,away_team,home_score,away_score,status
0,2026-09-24,A2,Netherlands,Germany,1.0,1.0,completed
1,2026-09-24,A2,Serbia,Greece,1.0,2.0,completed
2,2026-09-24,A4,Norway,Denmark,3.0,2.0,completed
3,2026-09-24,A4,Portugal,Wales,1.0,0.0,completed
4,2026-09-25,A1,Italy,Belgium,0.0,2.0,completed


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict, deque

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Historical matches
df = pd.read_csv("../data/results.csv")
df["date"] = pd.to_datetime(df["date"])

# Nations League 2026/27
nl_files = [
    p for p in Path("../data").glob("*.csv")
    if p.name.lower() != "results.csv"
]

nl = pd.read_csv(nl_files[0])
nl["date"] = pd.to_datetime(nl["date"])

print("Historical:", df.shape)
print("Nations League 2026/27:", nl.shape)

display(nl.tail(10))

Historical: (49547, 9)
Nations League 2026/27: (48, 7)


,date,group,home_team,away_team,home_score,away_score,status
38,2026-11-14,A4,Norway,Wales,NaN,NaN,upcoming
39,2026-11-14,A4,Portugal,Denmark,NaN,NaN,upcoming
40,2026-11-15,A1,Belgium,Italy,NaN,NaN,upcoming
41,2026-11-15,A1,France,Türkiye,NaN,NaN,upcoming
42,2026-11-15,A3,Croatia,Czechia,NaN,NaN,upcoming
43,2026-11-15,A3,Spain,England,NaN,NaN,upcoming
44,2026-11-16,A2,Germany,Netherlands,NaN,NaN,upcoming
45,2026-11-16,A2,Greece,Serbia,NaN,NaN,upcoming
46,2026-11-17,A4,Denmark,Norway,NaN,NaN,upcoming
47,2026-11-17,A4,Wales,Portugal,NaN,NaN,upcoming


In [4]:
# ==========================================
# TEAM FORM + ELO FEATURES
# ==========================================

def new_team():
    return {
        "elo": 1500.0,
        "points": deque(maxlen=5),
        "gd": deque(maxlen=5),
        "gf": deque(maxlen=5),
        "ga": deque(maxlen=5)
    }

teams = defaultdict(new_team)

def get_features(home, away, neutral=False):
    h = teams[home]
    a = teams[away]

    return [
        h["elo"],
        a["elo"],
        h["elo"] - a["elo"],
        np.mean(h["points"]) if h["points"] else 1.0,
        np.mean(a["points"]) if a["points"] else 1.0,
        np.mean(h["gd"]) if h["gd"] else 0.0,
        np.mean(a["gd"]) if a["gd"] else 0.0,
        np.mean(h["gf"]) if h["gf"] else 1.0,
        np.mean(a["gf"]) if a["gf"] else 1.0,
        np.mean(h["ga"]) if h["ga"] else 1.0,
        np.mean(a["ga"]) if a["ga"] else 1.0,
        int(bool(neutral))
    ]

def update_team(home, away, hs, aws):
    h = teams[home]
    a = teams[away]

    if hs > aws:
        result = 1.0
        hp, ap = 3, 0
    elif hs < aws:
        result = 0.0
        hp, ap = 0, 3
    else:
        result = 0.5
        hp, ap = 1, 1

    expected = 1 / (1 + 10 ** (-(h["elo"] - a["elo"]) / 400))

    margin = 1 + 0.5 * np.log1p(abs(hs - aws))
    change = 20 * margin * (result - expected)

    h["elo"] += change
    a["elo"] -= change

    h["points"].append(hp)
    a["points"].append(ap)

    h["gd"].append(hs - aws)
    a["gd"].append(aws - hs)

    h["gf"].append(hs)
    a["gf"].append(aws)

    h["ga"].append(aws)
    a["ga"].append(hs)


# ==========================================
# BUILD HISTORICAL TRAINING DATA
# ==========================================

hist = df.sort_values("date").reset_index(drop=True)

X = []
y = []

for _, r in hist.iterrows():

    X.append(
        get_features(
            r["home_team"],
            r["away_team"],
            r["neutral"]
        )
    )

    if r["home_score"] > r["away_score"]:
        y.append("H")
    elif r["home_score"] < r["away_score"]:
        y.append("A")
    else:
        y.append("D")

    update_team(
        r["home_team"],
        r["away_team"],
        int(r["home_score"]),
        int(r["away_score"])
    )

X = np.array(X)
y = np.array(y)

# Chronological split
split = int(len(X) * 0.80)

X_train = X[:split]
X_test = X[split:]
y_train = y[:split]
y_test = y[split:]

# ==========================================
# MODEL 1 — RANDOM FOREST
# ==========================================

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# ==========================================
# MODEL 2 — HISTOGRAM GRADIENT BOOSTING
# ==========================================

gb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

# ==========================================
# EVALUATION
# ==========================================

p_rf = rf.predict_proba(X_test)
p_gb = gb.predict_proba(X_test)

classes = rf.classes_

proba_test = (p_rf + p_gb) / 2
pred_test = classes[np.argmax(proba_test, axis=1)]

accuracy = accuracy_score(y_test, pred_test)

print(f"MODEL ACCURACY: {accuracy:.4f}\n")
print(classification_report(y_test, pred_test))
print("Confusion Matrix:")
print(confusion_matrix(y_test, pred_test))

# Save performance for Power BI
performance = pd.DataFrame({
    "metric": ["Accuracy"],
    "value": [accuracy]
})

performance.to_csv(
    "../results/model_performance.csv",
    index=False
)

# Retrain on ALL historical data
rf.fit(X, y)
gb.fit(X, y)

print("\nModels trained on all historical data.")

MODEL ACCURACY: 0.5925

              precision    recall  f1-score   support

           A       0.56      0.64      0.60      2888
           D       0.30      0.08      0.13      2305
           H       0.64      0.81      0.72      4717

    accuracy                           0.59      9910
   macro avg       0.50      0.51      0.48      9910
weighted avg       0.54      0.59      0.55      9910

Confusion Matrix:
[[1846  214  828]
 [ 815  189 1301]
 [ 643  237 3837]]

Models trained on all historical data.


In [5]:
# ==========================================
# TEAM FORM + ELO FEATURES
# ==========================================

def new_team():
    return {
        "elo": 1500.0,
        "points": deque(maxlen=5),
        "gd": deque(maxlen=5),
        "gf": deque(maxlen=5),
        "ga": deque(maxlen=5)
    }

teams = defaultdict(new_team)

def get_features(home, away, neutral=False):
    h = teams[home]
    a = teams[away]

    return [
        h["elo"],
        a["elo"],
        h["elo"] - a["elo"],
        np.mean(h["points"]) if h["points"] else 1.0,
        np.mean(a["points"]) if a["points"] else 1.0,
        np.mean(h["gd"]) if h["gd"] else 0.0,
        np.mean(a["gd"]) if a["gd"] else 0.0,
        np.mean(h["gf"]) if h["gf"] else 1.0,
        np.mean(a["gf"]) if a["gf"] else 1.0,
        np.mean(h["ga"]) if h["ga"] else 1.0,
        np.mean(a["ga"]) if a["ga"] else 1.0,
        int(bool(neutral))
    ]

def update_team(home, away, hs, aws):
    h = teams[home]
    a = teams[away]

    if hs > aws:
        result = 1.0
        hp, ap = 3, 0
    elif hs < aws:
        result = 0.0
        hp, ap = 0, 3
    else:
        result = 0.5
        hp, ap = 1, 1

    expected = 1 / (1 + 10 ** (-(h["elo"] - a["elo"]) / 400))

    margin = 1 + 0.5 * np.log1p(abs(hs - aws))
    change = 20 * margin * (result - expected)

    h["elo"] += change
    a["elo"] -= change

    h["points"].append(hp)
    a["points"].append(ap)

    h["gd"].append(hs - aws)
    a["gd"].append(aws - hs)

    h["gf"].append(hs)
    a["gf"].append(aws)

    h["ga"].append(aws)
    a["ga"].append(hs)


# ==========================================
# BUILD HISTORICAL TRAINING DATA
# ==========================================

hist = df.sort_values("date").reset_index(drop=True)

X = []
y = []

for _, r in hist.iterrows():

    X.append(
        get_features(
            r["home_team"],
            r["away_team"],
            r["neutral"]
        )
    )

    if r["home_score"] > r["away_score"]:
        y.append("H")
    elif r["home_score"] < r["away_score"]:
        y.append("A")
    else:
        y.append("D")

    update_team(
        r["home_team"],
        r["away_team"],
        int(r["home_score"]),
        int(r["away_score"])
    )

X = np.array(X)
y = np.array(y)

# Chronological split
split = int(len(X) * 0.80)

X_train = X[:split]
X_test = X[split:]
y_train = y[:split]
y_test = y[split:]

# ==========================================
# MODEL 1 — RANDOM FOREST
# ==========================================

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# ==========================================
# MODEL 2 — HISTOGRAM GRADIENT BOOSTING
# ==========================================

gb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

# ==========================================
# EVALUATION
# ==========================================

p_rf = rf.predict_proba(X_test)
p_gb = gb.predict_proba(X_test)

classes = rf.classes_

proba_test = (p_rf + p_gb) / 2
pred_test = classes[np.argmax(proba_test, axis=1)]

accuracy = accuracy_score(y_test, pred_test)

print(f"MODEL ACCURACY: {accuracy:.4f}\n")
print(classification_report(y_test, pred_test))
print("Confusion Matrix:")
print(confusion_matrix(y_test, pred_test))

# Save performance for Power BI
performance = pd.DataFrame({
    "metric": ["Accuracy"],
    "value": [accuracy]
})

performance.to_csv(
    "../results/model_performance.csv",
    index=False
)

# Retrain on ALL historical data
rf.fit(X, y)
gb.fit(X, y)

print("\nModels trained on all historical data.")

MODEL ACCURACY: 0.5925

              precision    recall  f1-score   support

           A       0.56      0.64      0.60      2888
           D       0.30      0.08      0.13      2305
           H       0.64      0.81      0.72      4717

    accuracy                           0.59      9910
   macro avg       0.50      0.51      0.48      9910
weighted avg       0.54      0.59      0.55      9910

Confusion Matrix:
[[1846  214  828]
 [ 815  189 1301]
 [ 643  237 3837]]

Models trained on all historical data.


In [6]:
# ==========================================
# UPDATE TEAM STATES WITH 2026/27 RESULTS
# ==========================================

played = nl[
    (nl["status"] == "completed") &
    nl["home_score"].notna() &
    nl["away_score"].notna()
].sort_values("date")

for _, r in played.iterrows():

    update_team(
        r["home_team"],
        r["away_team"],
        int(r["home_score"]),
        int(r["away_score"])
    )

print("Completed Nations League matches used:", len(played))


# ==========================================
# PREDICT ALL REMAINING MATCHES
# ==========================================

future = nl[nl["status"] == "upcoming"].copy()
future = future.sort_values("date").reset_index(drop=True)

predictions = []

for _, r in future.iterrows():

    features = np.array([
        get_features(
            r["home_team"],
            r["away_team"],
            False
        )
    ])

    p_rf = rf.predict_proba(features)[0]
    p_gb = gb.predict_proba(features)[0]

    p = (p_rf + p_gb) / 2

    home_p = p[list(classes).index("H")]
    draw_p = p[list(classes).index("D")]
    away_p = p[list(classes).index("A")]

    result = classes[np.argmax(p)]

    predictions.append({
        "date": r["date"],
        "group": r["group"],
        "home_team": r["home_team"],
        "away_team": r["away_team"],
        "prediction": result,
        "home_win_prob": home_p,
        "draw_prob": draw_p,
        "away_win_prob": away_p,
        "confidence": max(home_p, draw_p, away_p)
    })

pred_df = pd.DataFrame(predictions)

pred_df.to_csv(
    "../results/remaining_matches_predictions.csv",
    index=False
)

display(pred_df)

print(
    "\nRemaining matches predicted:",
    len(pred_df)
)

Completed Nations League matches used: 32


,date,group,home_team,away_team,prediction,home_win_prob,draw_prob,away_win_prob,confidence
0,2026-11-12,A1,Türkiye,Belgium,A,0.079723,0.176689,0.743588,0.743588
1,2026-11-12,A1,Italy,France,A,0.238692,0.309091,0.452218,0.452218
2,2026-11-12,A3,Czechia,Spain,A,0.038676,0.094957,0.866367,0.866367
3,2026-11-12,A3,England,Croatia,H,0.563170,0.219648,0.217182,0.563170
4,2026-11-13,A2,Netherlands,Greece,H,0.566683,0.315404,0.117913,0.566683
5,2026-11-13,A2,Serbia,Germany,A,0.166524,0.389391,0.444084,0.444084
6,2026-11-14,A4,Norway,Wales,H,0.504300,0.343289,0.152411,0.504300
7,2026-11-14,A4,Portugal,Denmark,H,0.536933,0.294410,0.168657,0.536933
8,2026-11-15,A1,Belgium,Italy,H,0.384375,0.333850,0.281775,0.384375
9,2026-11-15,A1,France,Türkiye,H,0.888770,0.087713,0.023518,0.888770



Remaining matches predicted: 16


In [7]:
# ==========================================
# KNOCKOUT STAGE PROJECTION
# ==========================================

# Recalculate rank safely from standings_df
standings_df = standings_df.sort_values(
    ["group", "projected_points", "goal_difference"],
    ascending=[True, False, False]
).copy()

standings_df["projected_rank"] = (
    standings_df.groupby("group").cumcount() + 1
)

qualifiers = standings_df[
    standings_df["projected_rank"] <= 2
].copy()

# ------------------------------------------
# Get group winners and runners-up
# ------------------------------------------

groups = sorted(qualifiers["group"].unique())

group_winners = {}
group_runners = {}

for g in groups:

    group_data = qualifiers[
        qualifiers["group"] == g
    ].sort_values("projected_rank")

    group_winners[g] = group_data.iloc[0]["team"]
    group_runners[g] = group_data.iloc[1]["team"]

print("GROUP WINNERS:")
print(group_winners)

print("\nGROUP RUNNERS-UP:")
print(group_runners)


# ------------------------------------------
# Function to predict a knockout match
# ------------------------------------------

def predict_one_match(home, away):

    features = np.array([
        get_features(home, away, False)
    ])

    p_rf = rf.predict_proba(features)[0]
    p_gb = gb.predict_proba(features)[0]

    p = (p_rf + p_gb) / 2

    home_p = p[list(classes).index("H")]
    draw_p = p[list(classes).index("D")]
    away_p = p[list(classes).index("A")]

    winner = home if home_p >= away_p else away

    return {
        "home_team": home,
        "away_team": away,
        "home_win_prob": home_p,
        "draw_prob": draw_p,
        "away_win_prob": away_p,
        "predicted_winner": winner
    }


# ------------------------------------------
# Projected quarter-final pairings
# ------------------------------------------

qf_pairs = [
    (group_winners[groups[0]], group_runners[groups[1]]),
    (group_winners[groups[1]], group_runners[groups[0]]),
    (group_winners[groups[2]], group_runners[groups[3]]),
    (group_winners[groups[3]], group_runners[groups[2]])
]


# ------------------------------------------
# Quarter-finals
# ------------------------------------------

qf_results = []
qf_winners = []

for i, (group_winner, runner_up) in enumerate(qf_pairs, 1):

    # First leg
    leg1 = predict_one_match(
        runner_up,
        group_winner
    )

    # Second leg
    leg2 = predict_one_match(
        group_winner,
        runner_up
    )

    # Simple two-leg probability aggregation
    group_winner_score = (
        leg1["away_win_prob"] +
        leg2["home_win_prob"]
    )

    runner_up_score = (
        leg1["home_win_prob"] +
        leg2["away_win_prob"]
    )

    qualified_team = (
        group_winner
        if group_winner_score >= runner_up_score
        else runner_up
    )

    qf_winners.append(qualified_team)

    qf_results.append({
        "quarter_final": f"QF{i}",
        "team_A": group_winner,
        "team_B": runner_up,
        "leg1_winner": leg1["predicted_winner"],
        "leg2_winner": leg2["predicted_winner"],
        "qualified_team": qualified_team
    })


qf_df = pd.DataFrame(qf_results)


# ------------------------------------------
# Semifinals
# ------------------------------------------

sf1 = predict_one_match(
    qf_winners[0],
    qf_winners[1]
)

sf2 = predict_one_match(
    qf_winners[2],
    qf_winners[3]
)

sf_df = pd.DataFrame([
    {
        "semi_final": "SF1",
        "team_A": sf1["home_team"],
        "team_B": sf1["away_team"],
        "predicted_winner": sf1["predicted_winner"],
        "home_win_prob": sf1["home_win_prob"],
        "draw_prob": sf1["draw_prob"],
        "away_win_prob": sf1["away_win_prob"]
    },
    {
        "semi_final": "SF2",
        "team_A": sf2["home_team"],
        "team_B": sf2["away_team"],
        "predicted_winner": sf2["predicted_winner"],
        "home_win_prob": sf2["home_win_prob"],
        "draw_prob": sf2["draw_prob"],
        "away_win_prob": sf2["away_win_prob"]
    }
])


# ------------------------------------------
# Final
# ------------------------------------------

final = predict_one_match(
    sf1["predicted_winner"],
    sf2["predicted_winner"]
)

final_df = pd.DataFrame([{
    "team_A": final["home_team"],
    "team_B": final["away_team"],
    "predicted_winner": final["predicted_winner"],
    "home_win_prob": final["home_win_prob"],
    "draw_prob": final["draw_prob"],
    "away_win_prob": final["away_win_prob"]
}])


# ------------------------------------------
# Champion
# ------------------------------------------

champion = final["predicted_winner"]

print("\n========== QUARTER-FINALS ==========")
display(qf_df)

print("\n========== SEMIFINALS ==========")
display(sf_df)

print("\n========== FINAL ==========")
display(final_df)

print("\n🏆 PREDICTED CHAMPION:", champion)


# ------------------------------------------
# Save for Power BI
# ------------------------------------------

qf_df.to_csv(
    "../results/quarter_finals_projection.csv",
    index=False
)

sf_df.to_csv(
    "../results/semifinals_projection.csv",
    index=False
)

final_df.to_csv(
    "../results/final_projection.csv",
    index=False
)

pd.DataFrame({
    "prediction": ["Champion"],
    "team": [champion]
}).to_csv(
    "../results/champion_prediction.csv",
    index=False
)

print("\nAll knockout files saved successfully.")

NameError: name 'standings_df' is not defined

In [ ]:
# ==========================================
# CELL 5 — FULL TOURNAMENT PROJECTION
# ==========================================

# ------------------------------------------
# 1. BUILD PROJECTED LEAGUE A STANDINGS
# ------------------------------------------

standings = []

for group, g in nl.groupby("group"):

    teams_group = sorted(
        set(g["home_team"]) | set(g["away_team"])
    )

    for team in teams_group:

        points = 0
        goals_for = 0
        goals_against = 0

        # Completed matches
        completed = g[
            (g["status"] == "completed") &
            (
                (g["home_team"] == team) |
                (g["away_team"] == team)
            )
        ]

        for _, r in completed.iterrows():

            if r["home_team"] == team:
                gf = int(r["home_score"])
                ga = int(r["away_score"])
            else:
                gf = int(r["away_score"])
                ga = int(r["home_score"])

            goals_for += gf
            goals_against += ga

            if gf > ga:
                points += 3
            elif gf == ga:
                points += 1

        # Expected points from future matches
        expected_points = 0

        future_team = pred_df[
            (pred_df["home_team"] == team) |
            (pred_df["away_team"] == team)
        ]

        for _, r in future_team.iterrows():

            if r["home_team"] == team:
                expected_points += (
                    3 * r["home_win_prob"] +
                    1 * r["draw_prob"]
                )
            else:
                expected_points += (
                    3 * r["away_win_prob"] +
                    1 * r["draw_prob"]
                )

        standings.append({
            "group": group,
            "team": team,
            "current_points": points,
            "expected_points": expected_points,
            "projected_points": points + expected_points,
            "goal_difference": goals_for - goals_against
        })


standings_df = pd.DataFrame(standings)

# Rank teams inside each group
standings_df = standings_df.sort_values(
    ["group", "projected_points", "goal_difference"],
    ascending=[True, False, False]
).reset_index(drop=True)

standings_df["projected_rank"] = (
    standings_df.groupby("group").cumcount() + 1
)

# League A = groups A1-A4
standings_df = standings_df[
    standings_df["group"].astype(str).str.startswith("A")
].copy()

print("========== PROJECTED LEAGUE A STANDINGS ==========")
display(standings_df)


# ------------------------------------------
# 2. GET TWO QUALIFIERS FROM EACH GROUP
# ------------------------------------------

qualifiers = standings_df[
    standings_df["projected_rank"] <= 2
].copy()

groups = sorted(qualifiers["group"].unique())

group_winners = {}
group_runners = {}

for group in groups:

    g = qualifiers[
        qualifiers["group"] == group
    ].sort_values("projected_rank")

    group_winners[group] = g.iloc[0]["team"]
    group_runners[group] = g.iloc[1]["team"]


print("========== GROUP WINNERS ==========")
print(group_winners)

print("\n========== GROUP RUNNERS-UP ==========")
print(group_runners)


# ------------------------------------------
# 3. MATCH PREDICTION FUNCTION
# ------------------------------------------

def predict_knockout(home, away):

    features = np.array([
        get_features(home, away, False)
    ])

    p_rf = rf.predict_proba(features)[0]
    p_gb = gb.predict_proba(features)[0]

    p = (p_rf + p_gb) / 2

    home_prob = p[list(classes).index("H")]
    draw_prob = p[list(classes).index("D")]
    away_prob = p[list(classes).index("A")]

    winner = home if home_prob >= away_prob else away

    return {
        "home_team": home,
        "away_team": away,
        "home_win_prob": home_prob,
        "draw_prob": draw_prob,
        "away_win_prob": away_prob,
        "predicted_winner": winner
    }


# ------------------------------------------
# 4. QUARTER-FINAL PROJECTION
# ------------------------------------------

# Model bracket:
# A1 winner vs A2 runner
# A2 winner vs A1 runner
# A3 winner vs A4 runner
# A4 winner vs A3 runner

qf_pairs = [
    (group_winners["A1"], group_runners["A2"]),
    (group_winners["A2"], group_runners["A1"]),
    (group_winners["A3"], group_runners["A4"]),
    (group_winners["A4"], group_runners["A3"])
]

qf_results = []
qf_winners = []

for i, (winner_seed, runner_seed) in enumerate(qf_pairs, 1):

    # Leg 1
    leg1 = predict_knockout(
        runner_seed,
        winner_seed
    )

    # Leg 2
    leg2 = predict_knockout(
        winner_seed,
        runner_seed
    )

    # Aggregate probabilities
    score_winner_seed = (
        leg1["away_win_prob"] +
        leg2["home_win_prob"]
    )

    score_runner_seed = (
        leg1["home_win_prob"] +
        leg2["away_win_prob"]
    )

    qualified = (
        winner_seed
        if score_winner_seed >= score_runner_seed
        else runner_seed
    )

    qf_winners.append(qualified)

    qf_results.append({
        "quarter_final": f"QF{i}",
        "team_A": winner_seed,
        "team_B": runner_seed,
        "leg1_winner": leg1["predicted_winner"],
        "leg2_winner": leg2["predicted_winner"],
        "qualified_team": qualified
    })


qf_df = pd.DataFrame(qf_results)

print("\n========== QUARTER-FINALS ==========")
display(qf_df)


# ------------------------------------------
# 5. SEMI-FINALS
# ------------------------------------------

sf1 = predict_knockout(
    qf_winners[0],
    qf_winners[1]
)

sf2 = predict_knockout(
    qf_winners[2],
    qf_winners[3]
)

sf_df = pd.DataFrame([
    {
        "semi_final": "SF1",
        "team_A": sf1["home_team"],
        "team_B": sf1["away_team"],
        "predicted_winner": sf1["predicted_winner"],
        "home_win_prob": sf1["home_win_prob"],
        "draw_prob": sf1["draw_prob"],
        "away_win_prob": sf1["away_win_prob"]
    },
    {
        "semi_final": "SF2",
        "team_A": sf2["home_team"],
        "team_B": sf2["away_team"],
        "predicted_winner": sf2["predicted_winner"],
        "home_win_prob": sf2["home_win_prob"],
        "draw_prob": sf2["draw_prob"],
        "away_win_prob": sf2["away_win_prob"]
    }
])

print("\n========== SEMI-FINALS ==========")
display(sf_df)


# ------------------------------------------
# 6. FINAL
# ------------------------------------------

final = predict_knockout(
    sf1["predicted_winner"],
    sf2["predicted_winner"]
)

final_df = pd.DataFrame([{
    "team_A": final["home_team"],
    "team_B": final["away_team"],
    "predicted_winner": final["predicted_winner"],
    "home_win_prob": final["home_win_prob"],
    "draw_prob": final["draw_prob"],
    "away_win_prob": final["away_win_prob"]
}])

print("\n========== FINAL ==========")
display(final_df)


# ------------------------------------------
# 7. CHAMPION
# ------------------------------------------

champion = final["predicted_winner"]

print("\n===================================")
print("🏆 PREDICTED CHAMPION:", champion)
print("===================================")


# ------------------------------------------
# 8. SAVE ALL RESULTS FOR POWER BI
# ------------------------------------------

standings_df.to_csv(
    "../results/league_a_projection.csv",
    index=False
)

qualifiers.to_csv(
    "../results/quarter_final_qualifiers.csv",
    index=False
)

qf_df.to_csv(
    "../results/quarter_finals_projection.csv",
    index=False
)

sf_df.to_csv(
    "../results/semifinals_projection.csv",
    index=False
)

final_df.to_csv(
    "../results/final_projection.csv",
    index=False
)

pd.DataFrame({
    "prediction": ["Champion"],
    "team": [champion]
}).to_csv(
    "../results/champion_prediction.csv",
    index=False
)

print("\nAll tournament files saved successfully.")

========== PROJECTED LEAGUE A STANDINGS ==========


,group,team,current_points,expected_points,projected_points,goal_difference,projected_rank
0,A1,France,7,6.055422,13.055422,2,1
1,A1,Belgium,6,5.105556,11.105556,4,2
2,A1,Italy,4,4.849252,8.849252,1,3
3,A1,Türkiye,0,0.648100,0.648100,-7,4
4,A2,Greece,7,3.687611,10.687611,2,1
5,A2,Netherlands,5,5.614107,10.614107,1,2
6,A2,Germany,4,4.623262,8.623262,1,3
7,A2,Serbia,0,2.209672,2.209672,-4,4
8,A3,Spain,9,6.963784,15.963784,6,1
9,A3,England,6,5.656799,11.656799,8,2


========== GROUP WINNERS ==========
{'A1': 'France', 'A2': 'Greece', 'A3': 'Spain', 'A4': 'Portugal'}

========== GROUP RUNNERS-UP ==========
{'A1': 'Belgium', 'A2': 'Netherlands', 'A3': 'England', 'A4': 'Norway'}

========== QUARTER-FINALS ==========


,quarter_final,team_A,team_B,leg1_winner,leg2_winner,qualified_team
0,QF1,France,Netherlands,France,France,France
1,QF2,Greece,Belgium,Belgium,Belgium,Belgium
2,QF3,Spain,Norway,Spain,Spain,Spain
3,QF4,Portugal,England,Portugal,England,England



========== SEMI-FINALS ==========


,semi_final,team_A,team_B,predicted_winner,home_win_prob,draw_prob,away_win_prob
0,SF1,France,Belgium,France,0.453838,0.313418,0.232744
1,SF2,Spain,England,Spain,0.556205,0.233778,0.210017



========== FINAL ==========


,team_A,team_B,predicted_winner,home_win_prob,draw_prob,away_win_prob
0,France,Spain,Spain,0.252671,0.366236,0.381093



🏆 PREDICTED CHAMPION: Spain

All tournament files saved successfully.


In [ ]:
from pathlib import Path

print("Notebook working directory:")
print(Path.cwd())

print("\nVariables available:")
for name in ["df", "nl", "rf", "gb", "pred_df", "standings_df", "qf_df", "sf_df", "final_df", "champion"]:
    print(name, "→", name in globals())

results_path = Path.cwd().parent / "results"

print("\nResults folder:")
print(results_path)

print("\nCurrent files:")
for f in results_path.glob("*.csv"):
    print(f.name, "→", f.stat().st_size, "bytes")

Notebook working directory:
c:\Users\PC\Desktop\NationsLeague_ML_Project\notebooks

Variables available:
df → True
nl → True
rf → True
gb → True
pred_df → True
standings_df → True
qf_df → True
sf_df → True
final_df → True
champion → True

Results folder:
c:\Users\PC\Desktop\NationsLeague_ML_Project\results

Current files:
champion_prediction.csv → 0 bytes
final_projection.csv → 0 bytes
league_a_projection.csv → 0 bytes
model_performance.csv → 0 bytes
quarter_finals_projection.csv → 0 bytes
quarter_final_qualifiers.csv → 0 bytes
remaining_matches_predictions.csv → 0 bytes
semifinals_projection.csv → 0 bytes


In [ ]:
from pathlib import Path
import pandas as pd

results_path = Path(r"C:\Users\PC\Desktop\NationsLeague_ML_Project\results")
results_path.mkdir(exist_ok=True)

# Export existing results
pred_df.to_csv(results_path / "remaining_matches_predictions.csv", index=False)
standings_df.to_csv(results_path / "league_a_projection.csv", index=False)
qualifiers.to_csv(results_path / "quarter_final_qualifiers.csv", index=False)
qf_df.to_csv(results_path / "quarter_finals_projection.csv", index=False)
sf_df.to_csv(results_path / "semifinals_projection.csv", index=False)
final_df.to_csv(results_path / "final_projection.csv", index=False)

pd.DataFrame({
    "metric": ["Accuracy"],
    "value": [accuracy]
}).to_csv(results_path / "model_performance.csv", index=False)

pd.DataFrame({
    "prediction": ["Champion"],
    "team": [champion]
}).to_csv(results_path / "champion_prediction.csv", index=False)

print("EXPORT COMPLETE\n")

for file in sorted(results_path.glob("*.csv")):
    print(f"{file.name}: {file.stat().st_size:,} bytes")

EXPORT COMPLETE

champion_prediction.csv: 33 bytes
final_projection.csv: 149 bytes
league_a_projection.csv: 972 bytes
model_performance.csv: 43 bytes
quarter_final_qualifiers.csv: 532 bytes
quarter_finals_projection.csv: 240 bytes
remaining_matches_predictions.csv: 2,730 bytes
semifinals_projection.csv: 249 bytes


In [8]:
# ============================================================
# FINAL FULL REFRESH
# Train -> Predict -> Standings -> Knockout -> Export
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, deque

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# ============================================================
# 1. LOAD ALL DATA
# ============================================================

historical = pd.read_csv("../data/results.csv")
nl = pd.read_csv("../data/nations_league_2026_27_league_a.csv")

historical["date"] = pd.to_datetime(historical["date"], errors="coerce")
nl["date"] = pd.to_datetime(nl["date"], errors="coerce")

if "neutral" not in historical.columns:
    historical["neutral"] = False

if "neutral" not in nl.columns:
    nl["neutral"] = False


# ============================================================
# 2. ADD ALL COMPLETED 2026/27 MATCHES TO TRAINING DATA
# ============================================================

current_completed = nl[
    (nl["status"].astype(str).str.lower() == "completed") &
    nl["home_score"].notna() &
    nl["away_score"].notna()
].copy()

columns = [
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "neutral"
]

historical = historical[columns].copy()
current_completed = current_completed[columns].copy()

training_df = pd.concat(
    [historical, current_completed],
    ignore_index=True
)

# Remove duplicate fixtures
training_df = (
    training_df
    .drop_duplicates(
        subset=["date", "home_team", "away_team"]
    )
    .sort_values("date")
    .reset_index(drop=True)
)

print("===================================================")
print("FULL DATA REFRESH")
print("===================================================")
print("Historical matches:", len(historical))
print("New completed 2026/27 matches:", len(current_completed))
print("Total training matches:", len(training_df))


# ============================================================
# 3. TEAM STATE
# ============================================================

def new_team():
    return {
        "elo": 1500.0,
        "points": deque(maxlen=5),
        "gd": deque(maxlen=5),
        "gf": deque(maxlen=5),
        "ga": deque(maxlen=5)
    }


teams = defaultdict(new_team)


def get_features_full(home, away, neutral=False):

    h = teams[home]
    a = teams[away]

    return [
        h["elo"],
        a["elo"],
        h["elo"] - a["elo"],

        np.mean(h["points"]) if h["points"] else 1.0,
        np.mean(a["points"]) if a["points"] else 1.0,

        np.mean(h["gd"]) if h["gd"] else 0.0,
        np.mean(a["gd"]) if a["gd"] else 0.0,

        np.mean(h["gf"]) if h["gf"] else 1.0,
        np.mean(a["gf"]) if a["gf"] else 1.0,

        np.mean(h["ga"]) if h["ga"] else 1.0,
        np.mean(a["ga"]) if a["ga"] else 1.0,

        int(bool(neutral))
    ]


def update_team_full(home, away, hs, aws):

    h = teams[home]
    a = teams[away]

    if hs > aws:
        result = 1.0
        hp, ap = 3, 0

    elif hs < aws:
        result = 0.0
        hp, ap = 0, 3

    else:
        result = 0.5
        hp, ap = 1, 1

    expected = 1 / (
        1 + 10 ** (-(h["elo"] - a["elo"]) / 400)
    )

    margin = 1 + 0.5 * np.log1p(abs(hs - aws))
    change = 20 * margin * (result - expected)

    h["elo"] += change
    a["elo"] -= change

    h["points"].append(hp)
    a["points"].append(ap)

    h["gd"].append(hs - aws)
    a["gd"].append(aws - hs)

    h["gf"].append(hs)
    a["gf"].append(aws)

    h["ga"].append(aws)
    a["ga"].append(hs)


# ============================================================
# 4. BUILD TRAINING FEATURES CHRONOLOGICALLY
# ============================================================

X_full = []
y_full = []

for _, r in training_df.iterrows():

    X_full.append(
        get_features_full(
            r["home_team"],
            r["away_team"],
            r["neutral"]
        )
    )

    if r["home_score"] > r["away_score"]:
        y_full.append("H")

    elif r["home_score"] < r["away_score"]:
        y_full.append("A")

    else:
        y_full.append("D")

    update_team_full(
        r["home_team"],
        r["away_team"],
        int(r["home_score"]),
        int(r["away_score"])
    )

X_full = np.array(X_full)
y_full = np.array(y_full)


# ============================================================
# 5. EVALUATE ON CHRONOLOGICAL 80/20 SPLIT
# ============================================================

split = int(len(X_full) * 0.80)

X_train = X_full[:split]
X_test = X_full[split:]

y_train = y_full[:split]
y_test = y_full[split:]

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

gb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

p_rf_test = rf.predict_proba(X_test)
p_gb_test = gb.predict_proba(X_test)

classes = rf.classes_

# Make class order consistent for both models
rf_classes = list(rf.classes_)
gb_classes = list(gb.classes_)

def aligned_proba(model_proba, model_classes, target_classes):
    result = np.zeros(len(target_classes))
    for i, c in enumerate(model_classes):
        result[list(target_classes).index(c)] = model_proba[i]
    return result


pred_test = []

for i in range(len(X_test)):

    prf = aligned_proba(
        p_rf_test[i],
        rf_classes,
        classes
    )

    pgb = aligned_proba(
        p_gb_test[i],
        gb_classes,
        classes
    )

    p = (prf + pgb) / 2

    pred_test.append(
        classes[np.argmax(p)]
    )

accuracy = accuracy_score(
    y_test,
    pred_test
)

print("\nMODEL ACCURACY:", round(accuracy, 4))
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        pred_test,
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        pred_test
    )
)


# ============================================================
# 6. RETRAIN BOTH MODELS ON ALL COMPLETED MATCHES
# ============================================================

rf.fit(X_full, y_full)
gb.fit(X_full, y_full)

print("\nModels retrained on ALL completed matches.")


# ============================================================
# 7. REBUILD TEAM STATES FROM ALL COMPLETED MATCHES
# ============================================================

teams = defaultdict(new_team)

for _, r in training_df.sort_values("date").iterrows():

    update_team_full(
        r["home_team"],
        r["away_team"],
        int(r["home_score"]),
        int(r["away_score"])
    )

print("Team states updated from all completed results.")


# ============================================================
# 8. PREDICT ALL UPCOMING MATCHES
# ============================================================

future = (
    nl[
        nl["status"].astype(str).str.lower() == "upcoming"
    ]
    .sort_values("date")
    .reset_index(drop=True)
)

predictions = []


def predict_match(home, away):

    features = np.array([
        get_features_full(
            home,
            away,
            False
        )
    ])

    prf = rf.predict_proba(features)[0]
    pgb = gb.predict_proba(features)[0]

    prf = aligned_proba(
        prf,
        rf_classes,
        classes
    )

    pgb = aligned_proba(
        pgb,
        gb_classes,
        classes
    )

    p = (prf + pgb) / 2

    home_prob = p[list(classes).index("H")]
    draw_prob = p[list(classes).index("D")]
    away_prob = p[list(classes).index("A")]

    result = classes[np.argmax(p)]

    winner = (
        home
        if home_prob >= away_prob
        else away
    )

    return {
        "home_team": home,
        "away_team": away,
        "prediction": result,
        "home_win_prob": home_prob,
        "draw_prob": draw_prob,
        "away_win_prob": away_prob,
        "confidence": max(
            home_prob,
            draw_prob,
            away_prob
        ),
        "predicted_winner": winner
    }


for _, r in future.iterrows():

    result = predict_match(
        r["home_team"],
        r["away_team"]
    )

    predictions.append({
        "date": r["date"].strftime("%Y-%m-%d"),
        "group": r["group"],
        "home_team": r["home_team"],
        "away_team": r["away_team"],
        "prediction": result["prediction"],
        "home_win_prob": result["home_win_prob"],
        "draw_prob": result["draw_prob"],
        "away_win_prob": result["away_win_prob"],
        "confidence": result["confidence"]
    })

pred_df = pd.DataFrame(predictions)

print("\n===================================================")
print("UPCOMING MATCH PREDICTIONS")
print("===================================================")

display(pred_df)


# ============================================================
# 9. PROJECTED LEAGUE A STANDINGS
# ============================================================

standings = []

for group, g in nl.groupby("group"):

    teams_group = sorted(
        set(g["home_team"]) |
        set(g["away_team"])
    )

    for team in teams_group:

        points = 0
        goals_for = 0
        goals_against = 0

        completed = g[
            (g["status"].astype(str).str.lower() == "completed") &
            (
                (g["home_team"] == team) |
                (g["away_team"] == team)
            )
        ]

        for _, r in completed.iterrows():

            if r["home_team"] == team:
                gf = int(r["home_score"])
                ga = int(r["away_score"])
            else:
                gf = int(r["away_score"])
                ga = int(r["home_score"])

            goals_for += gf
            goals_against += ga

            if gf > ga:
                points += 3
            elif gf == ga:
                points += 1

        expected_points = 0

        future_team = pred_df[
            (pred_df["home_team"] == team) |
            (pred_df["away_team"] == team)
        ]

        for _, r in future_team.iterrows():

            if r["home_team"] == team:

                expected_points += (
                    3 * r["home_win_prob"] +
                    r["draw_prob"]
                )

            else:

                expected_points += (
                    3 * r["away_win_prob"] +
                    r["draw_prob"]
                )

        standings.append({
            "group": group,
            "team": team,
            "current_points": points,
            "expected_points": expected_points,
            "projected_points": (
                points + expected_points
            ),
            "goal_difference": (
                goals_for - goals_against
            )
        })

standings_df = pd.DataFrame(
    standings
)

standings_df = (
    standings_df
    .sort_values(
        [
            "group",
            "projected_points",
            "goal_difference"
        ],
        ascending=[True, False, False]
    )
    .reset_index(drop=True)
)

standings_df["projected_rank"] = (
    standings_df
    .groupby("group")
    .cumcount() + 1
)

print("\n===================================================")
print("PROJECTED LEAGUE A STANDINGS")
print("===================================================")

display(standings_df)


# ============================================================
# 10. QUALIFIERS
# ============================================================

qualifiers = standings_df[
    standings_df["projected_rank"] <= 2
].copy()

groups = sorted(
    qualifiers["group"].unique()
)

group_winners = {}
group_runners = {}

for group in groups:

    g = (
        qualifiers[
            qualifiers["group"] == group
        ]
        .sort_values("projected_rank")
    )

    group_winners[group] = g.iloc[0]["team"]
    group_runners[group] = g.iloc[1]["team"]


# ============================================================
# 11. KNOCKOUT PREDICTION
# ============================================================

def predict_knockout(home, away):

    result = predict_match(
        home,
        away
    )

    return {
        "home_team": home,
        "away_team": away,
        "home_win_prob": result["home_win_prob"],
        "draw_prob": result["draw_prob"],
        "away_win_prob": result["away_win_prob"],
        "predicted_winner": result["predicted_winner"]
    }


# ============================================================
# 12. QUARTER-FINALS
# ============================================================

qf_pairs = [
    (
        group_winners["A1"],
        group_runners["A2"]
    ),
    (
        group_winners["A2"],
        group_runners["A1"]
    ),
    (
        group_winners["A3"],
        group_runners["A4"]
    ),
    (
        group_winners["A4"],
        group_runners["A3"]
    )
]

qf_results = []
qf_winners = []

for i, (winner_seed, runner_seed) in enumerate(
    qf_pairs,
    1
):

    leg1 = predict_knockout(
        runner_seed,
        winner_seed
    )

    leg2 = predict_knockout(
        winner_seed,
        runner_seed
    )

    winner_seed_score = (
        leg1["away_win_prob"] +
        leg2["home_win_prob"]
    )

    runner_seed_score = (
        leg1["home_win_prob"] +
        leg2["away_win_prob"]
    )

    qualified = (
        winner_seed
        if winner_seed_score >= runner_seed_score
        else runner_seed
    )

    qf_winners.append(
        qualified
    )

    qf_results.append({
        "quarter_final": f"QF{i}",
        "team_A": winner_seed,
        "team_B": runner_seed,
        "leg1_winner": leg1["predicted_winner"],
        "leg2_winner": leg2["predicted_winner"],
        "qualified_team": qualified
    })

qf_df = pd.DataFrame(
    qf_results
)


# ============================================================
# 13. SEMI-FINALS
# ============================================================

sf1 = predict_knockout(
    qf_winners[0],
    qf_winners[1]
)

sf2 = predict_knockout(
    qf_winners[2],
    qf_winners[3]
)

sf_df = pd.DataFrame([
    {
        "semi_final": "SF1",
        "team_A": sf1["home_team"],
        "team_B": sf1["away_team"],
        "predicted_winner": sf1["predicted_winner"],
        "home_win_prob": sf1["home_win_prob"],
        "draw_prob": sf1["draw_prob"],
        "away_win_prob": sf1["away_win_prob"]
    },
    {
        "semi_final": "SF2",
        "team_A": sf2["home_team"],
        "team_B": sf2["away_team"],
        "predicted_winner": sf2["predicted_winner"],
        "home_win_prob": sf2["home_win_prob"],
        "draw_prob": sf2["draw_prob"],
        "away_win_prob": sf2["away_win_prob"]
    }
])


# ============================================================
# 14. FINAL
# ============================================================

final = predict_knockout(
    sf1["predicted_winner"],
    sf2["predicted_winner"]
)

final_df = pd.DataFrame([{
    "team_A": final["home_team"],
    "team_B": final["away_team"],
    "predicted_winner": final["predicted_winner"],
    "home_win_prob": final["home_win_prob"],
    "draw_prob": final["draw_prob"],
    "away_win_prob": final["away_win_prob"]
}])


# ============================================================
# 15. CHAMPION
# ============================================================

champion = final["predicted_winner"]

champion_df = pd.DataFrame({
    "prediction": ["Champion"],
    "team": [champion]
})


# ============================================================
# 16. TROPHY PROBABILITIES
# ============================================================
#
# Approximate probability based on current projected
# quarter-final / semi-final / final pathway.
#

teams_all = sorted(
    set(nl["home_team"]) |
    set(nl["away_team"])
)

trophy_records = []

# Base approximation:
# current projected points are used as strength signal.
# Normalize projected contender strength within League A.

contenders = qualifiers[
    ["team", "projected_points"]
].copy()

if len(contenders) > 0:

    values = contenders["projected_points"].astype(float)

    weights = np.exp(
        values - values.max()
    )

    probabilities = (
        weights / weights.sum()
    ) * 100

    for team, probability in zip(
        contenders["team"],
        probabilities
    ):

        trophy_records.append({
            "team": team,
            "trophy_probability": probability
        })

trophy_df = pd.DataFrame(
    trophy_records
)

if not trophy_df.empty:

    trophy_df = trophy_df.sort_values(
        "trophy_probability",
        ascending=False
    )


# ============================================================
# 17. EXPORT EVERYTHING
# ============================================================

results_path = Path(
    "../results"
)

results_path.mkdir(
    exist_ok=True
)

pred_df.to_csv(
    results_path /
    "remaining_matches_predictions.csv",
    index=False
)

standings_df.to_csv(
    results_path /
    "league_a_projection.csv",
    index=False
)

qualifiers.to_csv(
    results_path /
    "quarter_final_qualifiers.csv",
    index=False
)

qf_df.to_csv(
    results_path /
    "quarter_finals_projection.csv",
    index=False
)

sf_df.to_csv(
    results_path /
    "semifinals_projection.csv",
    index=False
)

final_df.to_csv(
    results_path /
    "final_projection.csv",
    index=False
)

champion_df.to_csv(
    results_path /
    "champion_prediction.csv",
    index=False
)

if not trophy_df.empty:

    trophy_df.to_csv(
        results_path /
        "trophy_probabilities.csv",
        index=False
    )

pd.DataFrame({
    "metric": ["Accuracy"],
    "value": [accuracy]
}).to_csv(
    results_path /
    "model_performance.csv",
    index=False
)


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

print("\n===================================================")
print("✅ FULL REFRESH COMPLETE")
print("===================================================")

print(
    "Training matches:",
    len(training_df)
)

print(
    "New 2026/27 matches learned:",
    len(current_completed)
)

print(
    "Upcoming matches predicted:",
    len(pred_df)
)

print(
    "Projected Champion:",
    champion
)

print("\nFINAL:")
display(final_df)

print("\nFILES UPDATED:")
for file in sorted(results_path.glob("*.csv")):
    print(
        f"{file.name}: "
        f"{file.stat().st_size:,} bytes"
    )

FULL DATA REFRESH
Historical matches: 49547
New completed 2026/27 matches: 32
Total training matches: 49578

MODEL ACCURACY: 0.5968

Classification Report:
              precision    recall  f1-score   support

           A       0.56      0.64      0.60      2895
           D       0.32      0.09      0.15      2302
           H       0.65      0.82      0.72      4719

    accuracy                           0.60      9916
   macro avg       0.51      0.52      0.49      9916
weighted avg       0.55      0.60      0.55      9916

Confusion Matrix:
[[1847  221  827]
 [ 796  216 1290]
 [ 634  230 3855]]

Models retrained on ALL completed matches.
Team states updated from all completed results.

UPCOMING MATCH PREDICTIONS


,date,group,home_team,away_team,prediction,home_win_prob,draw_prob,away_win_prob,confidence
0,2026-11-12,A1,Türkiye,Belgium,A,0.078665,0.166800,0.754535,0.754535
1,2026-11-12,A1,Italy,France,A,0.233654,0.299600,0.466746,0.466746
2,2026-11-12,A3,Czechia,Spain,A,0.031894,0.095714,0.872392,0.872392
3,2026-11-12,A3,England,Croatia,H,0.550922,0.225398,0.223680,0.550922
4,2026-11-13,A2,Netherlands,Greece,H,0.561142,0.319189,0.119669,0.561142
5,2026-11-13,A2,Serbia,Germany,A,0.196510,0.343910,0.459581,0.459581
6,2026-11-14,A4,Norway,Wales,H,0.488577,0.333518,0.177905,0.488577
7,2026-11-14,A4,Portugal,Denmark,H,0.531745,0.305168,0.163087,0.531745
8,2026-11-15,A1,Belgium,Italy,H,0.399348,0.323502,0.277150,0.399348
9,2026-11-15,A1,France,Türkiye,H,0.904012,0.078796,0.017191,0.904012



PROJECTED LEAGUE A STANDINGS


,group,team,current_points,expected_points,projected_points,goal_difference,projected_rank
0,A1,France,10,4.490671,14.490671,5,1
1,A1,Belgium,6,3.951950,9.951950,1,2
2,A1,Italy,7,2.155516,9.155516,3,3
3,A1,Türkiye,0,0.533166,0.533166,-9,4
4,A2,Netherlands,8,3.439551,11.439551,2,1
5,A2,Greece,8,2.329498,10.329498,2,2
6,A2,Germany,5,2.981832,7.981832,1,3
7,A2,Serbia,0,1.964581,1.964581,-5,4
8,A3,Spain,12,4.529339,16.529339,7,1
9,A3,England,9,2.809937,11.809937,11,2



✅ FULL REFRESH COMPLETE
Training matches: 49578
New 2026/27 matches learned: 32
Upcoming matches predicted: 16
Projected Champion: Spain

FINAL:


,team_A,team_B,predicted_winner,home_win_prob,draw_prob,away_win_prob
0,France,Spain,Spain,0.221313,0.26719,0.511497



FILES UPDATED:
champion_prediction.csv: 33 bytes
final_projection.csv: 148 bytes
league_a_projection.csv: 977 bytes
model_performance.csv: 43 bytes
quarter_final_qualifiers.csv: 536 bytes
quarter_finals_projection.csv: 252 bytes
remaining_matches_predictions.csv: 1,854 bytes
semifinals_projection.csv: 251 bytes
trophy_probabilities.csv: 254 bytes
